In [12]:
import os, pandas as pd
from dotenv import load_dotenv

load_dotenv(".env")

MYSQL = {
    "host":     os.getenv("MYSQL_HOST", "localhost"),
    "port":     int(os.getenv("MYSQL_PORT", 3306)),
    "user":     os.getenv("MYSQL_USER", "root"),
    "password": os.getenv("MYSQL_PASSWORD"),
    "database": os.getenv("MYSQL_DATABASE", "logistics_db"),
    "charset":  "utf8mb4",
}

DATA_RAW  = os.path.join("..", "data", "raw")
TRAIN_CSV = os.path.join(DATA_RAW, "train.csv")

print("Config loaded from .env")
print(f"MySQL : {MYSQL['user']}@{MYSQL['host']}:{MYSQL['port']}/{MYSQL['database']}")

Config loaded from .env
MySQL : root@localhost:3306/logistics_db


In [13]:
# Load train.csv only

df = pd.read_csv(TRAIN_CSV)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

Shape: 45,593 rows × 20 columns


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26


In [14]:
# Clean

def clean_df(df):
    df = df.copy()

    # Strip whitespace safely column by column
    for col in df.columns:
        if df[col].dtype == object:
            try:
                df[col] = df[col].astype(str).str.strip()
                df[col] = df[col].replace("nan", None)
            except:
                pass

    # Fix "conditions Sunny" → "Sunny"
    df["Weatherconditions"] = df["Weatherconditions"].str.replace("conditions ", "", regex=False)

    # Fix "(min) 24" → 24
    df["Time_taken(min)"] = (
        df["Time_taken(min)"].astype(str)
        .str.extract(r"(\d+)")
        .astype(float)
    )

    # Coerce numerics
    for col in ["Delivery_person_Age", "Delivery_person_Ratings",
                "Vehicle_condition", "multiple_deliveries",
                "Restaurant_latitude", "Restaurant_longitude",
                "Delivery_location_latitude", "Delivery_location_longitude"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Parse date
    df["Order_Date"] = pd.to_datetime(
        df["Order_Date"], dayfirst=True, errors="coerce"
    ).dt.date

    df = df.dropna(subset=["Delivery_person_ID", "Time_taken(min)"])

    print(f"Cleaned: {len(df):,} rows")
    return df

df = clean_df(df)
df.head(3)

Cleaned: 45,593 rows


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,2,Snack,motorcycle,0.0,No,Urban,24.0
1,0xb379,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,2,Snack,scooter,1.0,No,Metropolitian,33.0
2,0x5d6d,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,0,Drinks,motorcycle,1.0,No,Urban,26.0


In [15]:
# Preview cleaned values

print("Weatherconditions:", df["Weatherconditions"].unique())
print("Road_traffic_density:", df["Road_traffic_density"].unique())
print("City:", df["City"].unique())
print("Festival:", df["Festival"].unique())
print("Type_of_vehicle:", df["Type_of_vehicle"].unique())

Weatherconditions: ['Sunny' 'Stormy' 'Sandstorms' 'Cloudy' 'Fog' 'Windy' 'NaN']
Road_traffic_density: ['High' 'Jam' 'Low' 'Medium' 'NaN']
City: ['Urban' 'Metropolitian' 'Semi-Urban' 'NaN']
Festival: ['No' 'Yes' 'NaN']
Type_of_vehicle: ['motorcycle' 'scooter' 'electric_scooter' 'bicycle']


In [16]:
# Connect & create tables

import pymysql

def get_connection():
    cfg = {k: v for k, v in MYSQL.items() if k != "database"}
    conn = pymysql.connect(**cfg)
    cur  = conn.cursor()
    cur.execute(
        f"CREATE DATABASE IF NOT EXISTS `{MYSQL['database']}` "
        f"CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;"
    )
    conn.select_db(MYSQL["database"])
    conn.commit()
    return conn

conn = get_connection()
cur  = conn.cursor()
print("Connected:", MYSQL["database"])

Connected: logistics_db


In [17]:
# Create tables

cur.execute("DROP TABLE IF EXISTS orders;")
cur.execute("DROP TABLE IF EXISTS drivers;")

cur.execute("""
    CREATE TABLE drivers (
        driver_id    VARCHAR(30)  PRIMARY KEY,
        age          INT,
        rating       DECIMAL(4,2),
        vehicle_type VARCHAR(30),
        created_at   DATETIME DEFAULT CURRENT_TIMESTAMP
    )
""")

cur.execute("""
    CREATE TABLE orders (
        order_id             VARCHAR(20) PRIMARY KEY,
        driver_id            VARCHAR(30),
        restaurant_lat       DECIMAL(10,6),
        restaurant_lng       DECIMAL(10,6),
        delivery_lat         DECIMAL(10,6),
        delivery_lng         DECIMAL(10,6),
        order_date           DATE,
        time_ordered         TIME,
        time_order_picked    TIME,
        weather_conditions   VARCHAR(30),
        road_traffic_density VARCHAR(20),
        vehicle_condition    INT,
        type_of_order        VARCHAR(20),
        type_of_vehicle      VARCHAR(20),
        multiple_deliveries  INT,
        festival             VARCHAR(5),
        city                 VARCHAR(20),
        time_taken_min       INT,
        FOREIGN KEY (driver_id) REFERENCES drivers(driver_id)
    )
""")
conn.commit()
print("Tables created: drivers, orders")

Tables created: drivers, orders


In [18]:
# Insert drivers

drivers = (
    df[["Delivery_person_ID", "Delivery_person_Age",
        "Delivery_person_Ratings", "Type_of_vehicle"]]
    .drop_duplicates(subset=["Delivery_person_ID"])
    .dropna(subset=["Delivery_person_ID"])
)

rows = [
    (
        row["Delivery_person_ID"],
        int(row["Delivery_person_Age"])       if pd.notna(row["Delivery_person_Age"])     else None,
        float(row["Delivery_person_Ratings"]) if pd.notna(row["Delivery_person_Ratings"]) else None,
        row["Type_of_vehicle"],
    )
    for _, row in drivers.iterrows()
]

cur.executemany(
    "INSERT IGNORE INTO drivers (driver_id, age, rating, vehicle_type) VALUES (%s,%s,%s,%s)",
    rows
)
conn.commit()
print(f"Inserted {cur.rowcount} drivers")

Inserted 1320 drivers


In [19]:
# Insert orders

def safe(v):
    try:
        if pd.isna(v): return None
    except: pass
    return v

rows = [
    (
        safe(row["ID"]),
        safe(row["Delivery_person_ID"]),
        safe(row["Restaurant_latitude"]),
        safe(row["Restaurant_longitude"]),
        safe(row["Delivery_location_latitude"]),
        safe(row["Delivery_location_longitude"]),
        safe(row["Order_Date"]),
        safe(row["Time_Orderd"]),
        safe(row["Time_Order_picked"]),
        safe(row["Weatherconditions"]),
        safe(row["Road_traffic_density"]),
        safe(row["Vehicle_condition"]),
        safe(row["Type_of_order"]),
        safe(row["Type_of_vehicle"]),
        safe(row["multiple_deliveries"]),
        safe(row["Festival"]),
        safe(row["City"]),
        safe(row["Time_taken(min)"]),
    )
    for _, row in df.iterrows()
]

cur.executemany("""
    INSERT IGNORE INTO orders (
        order_id, driver_id,
        restaurant_lat, restaurant_lng,
        delivery_lat, delivery_lng,
        order_date, time_ordered, time_order_picked,
        weather_conditions, road_traffic_density,
        vehicle_condition, type_of_order, type_of_vehicle,
        multiple_deliveries, festival, city, time_taken_min
    ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
""", rows)
conn.commit()
print(f"Inserted {cur.rowcount:,} orders")

Inserted 45,593 orders


In [20]:
# Verify

conn = get_connection()
cur  = conn.cursor()

cur.execute("SELECT COUNT(*) FROM drivers")
print(f"drivers : {cur.fetchone()[0]}")

cur.execute("SELECT COUNT(*) FROM orders")
print(f"orders  : {cur.fetchone()[0]:,}")

cur.execute("""
    SELECT order_id, driver_id, weather_conditions, city, time_taken_min
    FROM orders LIMIT 5
""")
for row in cur.fetchall():
    print(" ", row)

conn.close()
print("\nNotebook 2 complete!")

drivers : 1320
orders  : 45,593
  ('0x1000', 'HYDRES03DEL02', 'Sunny', 'Urban', 24)
  ('0x1001', 'BANGRES06DEL02', 'Windy', 'Metropolitian', 12)
  ('0x1002', 'COIMBRES07DEL01', 'Sunny', 'Metropolitian', 20)
  ('0x1003', 'SURRES05DEL02', 'Sunny', 'Metropolitian', 21)
  ('0x1004', 'BANGRES19DEL01', 'Cloudy', 'Metropolitian', 30)

Notebook 2 complete!
